# Квадрат, порядок и позиционные представления

### Семинар 11 · Продолжение примера из главы 9

Это авторская учебная глава по мотивам **Deep Learning with PyTorch Step-by-Step** Даниэля Годоя, а не глава из книги. Мы продолжаем её пример: по первым двум вершинам квадрата предсказываем следующие две. Пройдём путь от координат вершин и ручного расчёта одной головы до обучения моделей и рисунков их предсказаний. Все ячейки содержат готовые вычисления.

Сохраним данные и компактную архитектуру из `Chapter09.ipynb`, а способ задания позиции будем менять. Сначала модель вообще не получит явного позиционного кодирования. Затем добавим синусоиды, повороты RoPE, линейную поправку ALiBi и обучаемую таблицу T5. В конце разберём, почему отсутствие позиционного кодирования не равнозначно отсутствию информации о порядке.

Рядом с ноутбуком нужна папка `position_square`: генератор квадратов, `StepByStep` и функции рисунков. Как в репозитории книги, технические функции вынесены из повествования. Используются NumPy, PyTorch, Matplotlib, torchvision и TensorBoard из окружения курса. При локальном запуске используются файлы рядом с ноутбуком. В Colab первая ячейка скачивает пять небольших файлов из фиксированного коммита GitHub и проверяет их SHA-256; данные генерируются на месте. Полный проход обучает пять небольших моделей по 100 эпох; время зависит от устройства, для таких маленьких матриц CPU может быть быстрее GPU.

In [ ]:
import copy
import math
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from matplotlib_inline.backend_inline import set_matplotlib_formats

# В Colab загружаем тот же набор функций Годоя из фиксированной версии.
if "google.colab" in sys.modules:
    import hashlib
    import tempfile
    from urllib.request import urlopen
    helper_revision = "a3001276af17c7770026e2fd1bc54a93063469b9"
    helper_hashes = {'LICENSE-Godoy.txt': '991e47263b5937accf877c58cca15c390520b5d99edba3885dcb299b89876e3c', 'square_sequences.py': '3df5eeb61555c7aab4ee5a33a489d4fe009af6d728da945f00f26c126fc7e24d', '__init__.py': '5abad5c8115317859701d9cfa3d04c8cc924036c2471ebffda901d495a0ee6ff', 'plots.py': 'a2efb6c36f44780ebda8fce3b96a774ec33a70b71192496c7063430009f49de5', 'stepbystep.py': 'd200a79d4dd1b50a9f20ae8fd01485af583799737d6033d546961e849b199d4f'}
    helper_root = Path(tempfile.mkdtemp(prefix="square-positions-"))
    helper_dir = helper_root / "position_square"
    helper_dir.mkdir()
    for filename, expected in helper_hashes.items():
        url = ("https://raw.githubusercontent.com/SergeyMalashenko/2026_ML3/"
               + helper_revision + "/notebooks/11_gpt/position_square/" + filename)
        with urlopen(url, timeout=60) as response:
            content = response.read()
        if hashlib.sha256(content).hexdigest() != expected:
            raise RuntimeError("Не совпала контрольная сумма: " + filename)
        (helper_dir / filename).write_bytes(content)
    # Меняем путь только после проверки всех файлов; повторный запуск безопасен.
    for module in list(sys.modules):
        if module == "position_square" or module.startswith("position_square."):
            del sys.modules[module]
    sys.path.insert(0, str(helper_root))

# Jupyter может запускать ядро из папки ноутбука или из корня курса.
for base in [Path.cwd(), Path.cwd() / "notebooks/11_gpt"]:
    if (base / "position_square").is_dir():
        sys.path.append(str(base.resolve()))
        break
from position_square.square_sequences import generate_sequences
from position_square.stepbystep import StepByStep
from position_square.plots import plot_data, sequence_pred, plot_matrix, plot_attention, plot_comparison, add_arrow

set_matplotlib_formats("svg", "retina")
# Тот же стиль, что в StepByStep Годоя, без переопределения размеров шрифта.
plt.style.use("fivethirtyeight")
use_cpu = False
if not use_cpu and torch.backends.mps.is_built() and torch.backends.mps.is_available():
    device = "mps"
elif not use_cpu and torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
# Небольшие матрицы выгоднее считать в одном потоке CPU.
torch.set_num_threads(1)
print("Устройство:", device)

### Содержание

1. [Генерация данных](#section-1)
2. [Энкодер, декодер и одна голова внимания](#section-2)
3. [Модель без явного позиционного кодирования](#section-3)
4. [Синусоидальное кодирование](#section-4)
5. [RoPE: поворачиваем запросы и ключи](#section-5)
6. [ALiBi: меняем оценки за счёт расстояния](#section-6)
7. [T5: обучаем позиционную поправку](#section-7)
8. [Обучение на одних и тех же квадратах](#section-8)
9. [Что показывает сравнение](#section-9)
10. [NoPE и информация в причинной маске](#section-10)
11. [Другие относительные представления](#section-11)
12. [Источники и устройство файлов](#section-12)

<a id="section-1"></a>

## 1. Генерация данных


### Четыре вершины, два направления

У квадрата четыре базовые вершины: $(-1,-1)$, $(-1,1)$, $(1,1)$, $(1,-1)$. Генератор Годоя случайно выбирает начало обхода и направление, затем добавляет независимый гауссов шум со стандартным отклонением 0.1 к координатам. Поэтому геометрия узнаваема, но примеры не идентичны.

Буквы A, B, C, D обозначают **порядок посещения**, а не закреплённые углы плоскости. A и B — вход; C и D — то, что мы хотим предсказать. Метки направления нужны только для рисунков, модель их не получает. Не будем путать координаты вершины на плоскости с её позицией в последовательности: один и тот же угол квадрата может оказаться первым, вторым, третьим или четвёртым.

На графиках сохраняем обозначения книги: номера **1, 2, 3, 4** внутри чёрных кружков соответствуют **A, B, C, D** в тексте.

In [ ]:
points, directions = generate_sequences(n=256, seed=13)
fig = plot_data(points, directions)

### Один квадрат без шума

Для разбора вычислений оставим один точный квадрат, как в книге. Первая размерность тензора — номер примера, вторая — позиция, третья — две координаты. `source_seq` имеет форму $(1,2,2)$, `target_seq` — такую же. Здесь нет текстового токенизатора: входной элемент последовательности — вершина.

In [ ]:
full_seq = torch.tensor([[-1, -1], [-1, 1], [1, 1], [1, -1]]).float().view(1, 4, 2)
source_seq = full_seq[:, :2] # first two corners
target_seq = full_seq[:, 2:] # last two corners
source_seq, target_seq

### Подготовка данных

Сохраним обучающую выборку книги: 256 последовательностей, seed=13. Вторая выборка содержит 128 последовательностей с seed=19. Имя `test` оставлено как у Годоя, но эта выборка проверяется после каждой эпохи и используется для выбора настроек: по роли это **валидация**, не независимый финальный тест.

При обучении передаём всю четвёрку, чтобы сформировать сдвинутый вход декодера. При проверке модели доступны только A и B. Во всех вариантах используем одни и те же массивы.

In [ ]:
full_train = torch.as_tensor(np.asarray(points)).float()
target_train = full_train[:, 2:]
train_data = TensorDataset(full_train, target_train)

In [ ]:
test_points, test_directions = generate_sequences(n=128, seed=19)
full_test = torch.as_tensor(np.asarray(test_points)).float()
source_test = full_test[:, :2]
target_test = full_test[:, 2:]
test_data = TensorDataset(source_test, target_test)

In [ ]:
def make_loaders():
    generator = torch.Generator().manual_seed(42)
    train_loader = DataLoader(train_data, batch_size=16, shuffle=True, generator=generator)
    test_loader = DataLoader(test_data, batch_size=16)
    return train_loader, test_loader

train_loader, test_loader = make_loaders()
full_train.shape, source_test.shape, target_test.shape

<a id="section-2"></a>

## 2. Энкодер, декодер и одна голова внимания


### Сначала — оценки, затем веса, затем контекст

По A и B надо предсказать C, затем D. При обучении декодеру передаётся сдвинутая пара $(B,C)$: по B он учится выдавать C, по известной C — выдавать D. Это **teacher forcing**, обучение с подстановкой правильного предыдущего элемента. Маска запретит строке B использовать C, иначе ответ первого шага уже окажется во входе.

В одной голове внимания запрос $q_i$ сравнивается с ключами $k_j$. Оценки совместимости делятся на $\sqrt{d_k}$ и преобразуются softmax в веса $\alpha_{ij}$. С этими весами складываются **значения** $v_j$:
$$s_{ij}=\frac{q_i^\top k_j}{\sqrt{d_k}},\qquad
\alpha_{ij}=\frac{e^{s_{ij}}}{\sum_\ell e^{s_{i\ell}}},\qquad c_i=\sum_j\alpha_{ij}v_j.$$
Это веса по вершинам, не вероятности следующего слова. Пока возьмём координаты B и C как уже готовые запросы, ключи и значения. Такая ручная проверка показывает механику; у обучаемой модели ниже будут отдельные матрицы преобразования.

In [ ]:
shifted_seq = torch.cat([source_seq[:, -1:], target_seq[:, :-1]], dim=1)
shifted_seq

In [ ]:
query = keys = values = shifted_seq
products = torch.bmm(query, keys.transpose(1, 2)) / math.sqrt(2)
target_mask = torch.ones(1, 2, 2, dtype=torch.bool).tril()
alphas = products.masked_fill(~target_mask, -torch.inf).softmax(-1)
context_vector = torch.bmm(alphas, values)
products, alphas, context_vector

Первая строка получает B и предсказывает C. Вторая при обучении получает **известную** C и предсказывает D — это teacher forcing. Причинная маска запрещает первой строке увидеть C. При генерации второй строке будет доступна уже **предсказанная** C: ошибка первого шага может перейти на второй.

### Класс Attention

Оставим интерфейс Годоя: `init_keys`, `score_function`, `forward`, а также `alphas` для рисунков. Позиционные методы будут менять функцию оценки. Маска и взвешенная сумма останутся общими.

In [ ]:
class Attention(nn.Module):
    def __init__(self, hidden_dim, input_dim=None, proj_values=False):
        super().__init__()
        self.d_k = hidden_dim
        self.input_dim = hidden_dim if input_dim is None else input_dim
        self.proj_values = proj_values
        # Affine transformations for Q, K, and V
        self.linear_query = nn.Linear(self.input_dim, hidden_dim)
        self.linear_key = nn.Linear(self.input_dim, hidden_dim)
        self.linear_value = nn.Linear(self.input_dim, hidden_dim)
        self.alphas = None

    def init_keys(self, keys):
        self.keys = keys
        self.proj_keys = self.linear_key(self.keys)
        self.values = self.linear_value(self.keys) \
                      if self.proj_values else self.keys

    def score_function(self, query):
        proj_query = self.linear_query(query)
        # scaled dot product
        # N, 1, H x N, H, L -> N, 1, L
        dot_products = torch.bmm(proj_query, self.proj_keys.permute(0, 2, 1))
        scores =  dot_products / np.sqrt(self.d_k)
        return scores

    def forward(self, query, mask=None):
        # Query is batch-first N, 1, H
        scores = self.score_function(query) # N, 1, L
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        alphas = F.softmax(scores, dim=-1) # N, 1, L
        self.alphas = alphas.detach()

        # N, 1, L x N, L, H -> N, 1, H
        context = torch.bmm(alphas, self.values)
        return context

### Несколько голов

В реализации этой главы книги каждая голова имеет `d_model` координат; результаты объединяются и преобразуются обратно в `d_model`. При трёх головах и `d_model=2` объединённый вектор имеет шесть координат. Это отличается от распространённой реализации, где общую размерность делят между головами.

К конструктору добавлен только выбор `attention_cls`: он позволит заменить правило оценивания, сохранив остальную модель.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, n_heads, d_model, input_dim=None, proj_values=True, attention_cls=Attention):
        super().__init__()
        self.linear_out = nn.Linear(n_heads * d_model, d_model)
        self.attn_heads = nn.ModuleList([attention_cls(d_model, 
                                                   input_dim=input_dim, 
                                                   proj_values=proj_values) 
                                         for _ in range(n_heads)])

    def init_keys(self, key):
        for attn in self.attn_heads:
            attn.init_keys(key)

    @property
    def alphas(self):
        # Shape: n_heads, N, 1, L (source)
        return torch.stack([attn.alphas for attn in self.attn_heads], dim=0)

    def output_function(self, contexts):
        # N, 1, n_heads * D
        concatenated = torch.cat(contexts, axis=-1)
        # Linear transf. to go back to original dimension
        out = self.linear_out(concatenated) # N, 1, D
        return out

    def forward(self, query, mask=None):
        contexts = [attn(query, mask=mask) for attn in self.attn_heads]
        out = self.output_function(contexts)
        return out

### Энкодер

Обе известные вершины A и B доступны энкодеру одновременно. Его слой самовнимания не имеет причинной маски. После объединения голов применяется небольшая полносвязная сеть с ReLU.

In [ ]:
class EncoderSelfAttn(nn.Module):
    def __init__(self, n_heads, d_model, ff_units, n_features=None, attention_cls=Attention):
        super().__init__()
        self.n_heads = n_heads
        self.d_model = d_model
        self.ff_units = ff_units
        self.n_features = n_features
        self.self_attn_heads = MultiHeadAttention(n_heads, d_model, input_dim=n_features, attention_cls=attention_cls)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ff_units),
            nn.ReLU(),
            nn.Linear(ff_units, d_model),
        )

    def forward(self, query, mask=None):
        self.self_attn_heads.init_keys(query)
        att = self.self_attn_heads(query, mask)
        out = self.ffn(att)
        return out

### Декодер

Сначала декодер собирает сведения из собственного префикса через слой самовнимания с маской. Затем слой перекрёстного внимания читает выходы энкодера. В нашем сравнении позиционные методы заменяют **только самовнимание** энкодера и декодера; перекрёстное внимание во всех вариантах остаётся обычным. Нельзя механически считать разность локальных индексов двух разных последовательностей универсальным расстоянием.

In [ ]:
class DecoderSelfAttn(nn.Module):
    def __init__(self, n_heads, d_model, ff_units, n_features=None, attention_cls=Attention):
        super().__init__()
        self.n_heads = n_heads
        self.d_model = d_model
        self.ff_units = ff_units
        self.n_features = d_model if n_features is None else n_features
        self.self_attn_heads = MultiHeadAttention(n_heads, d_model, input_dim=self.n_features, attention_cls=attention_cls)
        self.cross_attn_heads = MultiHeadAttention(n_heads, d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ff_units),
            nn.ReLU(),
            nn.Linear(ff_units, self.n_features),
        )

    def init_keys(self, states):
        self.cross_attn_heads.init_keys(states)

    def forward(self, query, source_mask=None, target_mask=None):
        self.self_attn_heads.init_keys(query)
        att1 = self.self_attn_heads(query, target_mask)
        att2 = self.cross_attn_heads(att1, source_mask)
        out = self.ffn(att2)
        return out

### Собираем энкодер и декодер

Следующий класс из главы 9 различает обучение и генерацию через `model.train()` и `model.eval()`. Это не смена весов: меняется способ формирования входа декодера. При проверке он начинает с B, получает $\hat C$, затем подаёт $(B,\hat C)$ и получает $\hat D$.

In [ ]:
class EncoderDecoderSelfAttn(nn.Module):
    def __init__(self, encoder, decoder, input_len, target_len):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.input_len = input_len
        self.target_len = target_len
        self.trg_masks = self.subsequent_mask(self.target_len)

    @staticmethod
    def subsequent_mask(size):
        attn_shape = (1, size, size)
        subsequent_mask = (1 - torch.triu(torch.ones(attn_shape), diagonal=1))
        return subsequent_mask

    def encode(self, source_seq, source_mask):
        # Encodes the source sequence and uses the result
        # to initialize the decoder
        encoder_states = self.encoder(source_seq, source_mask)
        self.decoder.init_keys(encoder_states)

    def decode(self, shifted_target_seq, source_mask=None, target_mask=None):
        # Decodes/generates a sequence using the shifted (masked)
        # target sequence - used in TRAIN mode
        outputs = self.decoder(shifted_target_seq, 
                               source_mask=source_mask,
                               target_mask=target_mask)
        return outputs

    def predict(self, source_seq, source_mask):
        # Decodes/generates a sequence using one input
        # at a time - used in EVAL mode
        inputs = source_seq[:, -1:]
        for i in range(self.target_len):
            out = self.decode(inputs, source_mask, self.trg_masks[:, :i+1, :i+1])
            out = torch.cat([inputs, out[:, -1:, :]], dim=-2)
            inputs = out.detach()
        outputs = inputs[:, 1:, :]
        return outputs

    def forward(self, X, source_mask=None):
        # Sends the mask to the same device as the inputs
        self.trg_masks = self.trg_masks.type_as(X).bool()
        # Slices the input to get source sequence
        source_seq = X[:, :self.input_len, :]
        # Encodes source sequence AND initializes decoder
        self.encode(source_seq, source_mask)
        if self.training:
            # Slices the input to get the shifted target seq
            shifted_target_seq = X[:, self.input_len-1:-1, :]
            # Decodes using the mask to prevent cheating
            outputs = self.decode(shifted_target_seq, source_mask, self.trg_masks)
        else:
            # Decodes using its own predictions
            outputs = self.predict(source_seq, source_mask)

        return outputs

<a id="section-3"></a>

## 3. Модель без явного позиционного кодирования


### Что означает «без позиции»

Класс `Attention` не получает номера позиций. Без маски перестановка строк входа просто переставляет строки результата: это **эквивариантность к перестановке**, а не неизменность всего выходного тензора.

Проверим это у энкодера, поменяв A и B местами и вернув порядок выходов назад. Значения совпадут с точностью вычислений. Это свойство одного энкодера; весь encoder–decoder не обязан давать тот же прогноз, поскольку последняя входная вершина отдельно поступает в декодер.

In [ ]:
torch.manual_seed(43)
encself = EncoderSelfAttn(n_heads=3, d_model=2, ff_units=10, n_features=2)
with torch.no_grad():
    original = encself(source_seq)
    permuted = encself(source_seq.flip(1)).flip(1)
(original - permuted).abs().max()

Ниже назовём этот вариант **NoPE-контроль**: ни синусоид, ни относительных поправок. Однако это компактный encoder–decoder Годоя, а не полная decoder-only архитектура из статьи о NoPE. В нём нет остаточных связей и LayerNorm; энкодер двунаправленный, декодер причинный. Эти свойства сохраняются во всех пяти вариантах опыта.

Сама по себе неудачная траектория после одного обучения ещё не доказывает, что модель неспособна решить задачу. В книге отдельно отмечено влияние начального состояния генератора случайных чисел. Поэтому ниже мы показываем получившиеся результаты, а не заранее обещанные «треугольники».

<a id="section-4"></a>

## 4. Синусоидальное кодирование


### Прибавляем вектор позиции

Годой использует синусы и косинусы с разными частотами. Для пары координат с номером $r$:
$$\omega_r=10000^{-2r/d_{model}},\quad PE_{p,2r}=\sin(p\omega_r),\quad PE_{p,2r+1}=\cos(p\omega_r).$$
При двух координатах остаётся одна частота $\omega_0=1$. В реализации книги вход сначала умножается на $\sqrt{d_{model}}$, затем прибавляется PE. Сохраним и это масштабирование: вариант Sinusoidal воспроизводит именно этот приём книги, а не изолированное прибавление PE при неизменном масштабе.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, max_len, d_model):
        super().__init__()
        self.d_model = d_model
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).float().unsqueeze(1)
        angular_speed = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * angular_speed) # even dimensions
        pe[:, 1::2] = torch.cos(position * angular_speed) # odd dimensions
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        # x is N, L, D
        # pe is 1, maxlen, D
        scaled_x = x * np.sqrt(self.d_model)
        encoded = scaled_x + self.pe[:, :x.size(1), :]
        return encoded

In [ ]:
posenc = PositionalEncoding(max_len=4, d_model=2)
posenc.pe, posenc(source_seq)

Позиционный вектор добавляется **до** вычисления запросов, ключей и значений. В упрощённом случае единичных матриц произведение $(h_i+p_i)^\top(h_j+p_j)$ содержит четыре слагаемых: содержание–содержание, два смешанных вклада и позиция–позиция. Поэтому синусоидальное сложение не тождественно относительному повороту готовых Q и K.

<a id="section-5"></a>

## 5. RoPE: поворачиваем запросы и ключи


### Положение как угол поворота

RoPE действует на уже вычисленные запросы и ключи. В каждой паре координат применяется поворот $R(p\omega)$; значения V остаются прежними. Для столбца из двух координат матрица имеет вид
$$R(\phi)=\begin{pmatrix}\cos\phi&-\sin\phi\\\sin\phi&\cos\phi\end{pmatrix}.$$
Она ортогональна: $R(\phi)^\top=R(-\phi)$ и $R(\alpha)R(\beta)=R(\alpha+\beta)$. Поэтому умножение поворотов даёт
$$(R(i\omega)q)^\top R(j\omega)k=q^\top R((j-i)\omega)k.$$
Общий сдвиг $i$ и $j$ исчезает из разности. Поворот сохраняет длину каждого вектора, но меняет угол между запросом и ключом. В двумерной голове снова одна частота: один радиан за позицию.

In [ ]:
def rotate_pairs(x, offset=0):
    positions = torch.arange(offset, offset + x.size(1), device=x.device, dtype=x.dtype)
    frequencies = torch.exp(torch.arange(0, x.size(-1), 2, device=x.device, dtype=x.dtype)
                            * (-math.log(10000.0) / x.size(-1)))
    angles = positions[:, None] * frequencies[None, :]
    even, odd = x[..., 0::2], x[..., 1::2]
    rotated = torch.stack([even * angles.cos() - odd * angles.sin(),
                           even * angles.sin() + odd * angles.cos()], dim=-1)
    return rotated.flatten(-2)

class AttentionRoPE(Attention):
    def score_function(self, query):
        q = rotate_pairs(self.linear_query(query))
        k = rotate_pairs(self.proj_keys)
        return torch.bmm(q, k.transpose(1, 2)) / math.sqrt(self.d_k)

In [ ]:
q = shifted_seq.clone()
k = shifted_seq.clone()
score = rotate_pairs(q) @ rotate_pairs(k).transpose(1, 2)
shifted_score = rotate_pairs(q, offset=7) @ rotate_pairs(k, offset=7).transpose(1, 2)
score, (score - shifted_score).abs().max()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(8, 4))
for ax, offset in zip(axs, [0, 7]):
    xy = rotate_pairs(shifted_seq, offset)[0].numpy()
    ax.add_patch(plt.Circle((0, 0), math.sqrt(2), fill=False, color='k', lw=2))
    for name, p, color in zip(["B", "C"], xy, ['r', 'k']):
        line = ax.plot([0, p[0]], [0, p[1]], lw=0)[0]
        add_arrow(line, position=0, lw=2, color=color, text=name,
                  size=12, text_offset=(.06, .06))
    ax.set(xlim=(-1.9, 1.9), ylim=(-1.9, 1.9), aspect="equal",
           title=f"Позиции {offset}, {offset+1}", xlabel=r"$x_0$", ylabel=r"$x_1$")
fig.tight_layout()
plt.show()

На рисунке вращаются координаты вершин как демонстрационные Q/K, не физический квадрат целиком. У обучаемой головы вращаются результаты линейных преобразований. Одинаковый относительный угол объясняет инвариантность оценки к общему сдвигу; он не означает, что оценка монотонно убывает с расстоянием или что модель уже умеет работать на любой длине.

<a id="section-6"></a>

## 6. ALiBi: меняем оценки за счёт расстояния


### Позиция меняет оценку, но не векторы

Для причинной головы ALiBi вычитает $m_h(i-j)$ из масштабированной оценки разрешённого ключа. Коэффициент $m_h>0$ фиксирован для головы. Здесь сначала показана одна голова с $m_h=0.5$:
$$s_{ij}=q_i^\top k_j/\sqrt{d_k}-m_h(i-j),\quad j\leq i.$$
У энкодера нет ограничения $j\leq i$, поэтому для него в этом опыте используем симметричную адаптацию $-m_h|i-j|$. Это наш выбор для двунаправленного энкодера, не причинная схема исходной статьи. На разрешённой части декодера обе записи совпадают.

In [ ]:
class AttentionALiBi(Attention):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.register_buffer("slope", torch.tensor(0.5))

    def score_function(self, query):
        scores = super().score_function(query)
        i = torch.arange(query.size(1), device=query.device)[:, None]
        j = torch.arange(self.proj_keys.size(1), device=query.device)[None, :]
        return scores - self.slope * (i-j).abs()

In [ ]:
distance = torch.arange(2)[:, None] - torch.arange(2)[None, :]
alibi_scores = products - 0.5 * distance.abs()
alibi_alphas = alibi_scores.masked_fill(~target_mask, -torch.inf).softmax(-1)
fig = plot_matrix(alibi_alphas[0], "ALiBi: веса для B и C", ["B", "C"])

Первая строка по-прежнему имеет единственный доступный ключ B и вес 1. Во второй строке штраф уменьшает относительный вес B по сравнению с C. Это предпочтение близкого контекста, а не безусловный запрет на далёкий: содержательная оценка может перевесить штраф.

Почему линейная поправка действует как предпочтение расстояния? После экспоненты множители разделяются:
$$\alpha_{ij}\propto \exp(q_i^\top k_j/\sqrt{d_k})\,\exp(-m_h|i-j|).$$
При равных содержательных оценках каждый дополнительный шаг уменьшает ненормированный вес в $e^{m_h}$ раз. Нормировка softmax затем делает сумму разрешённых весов равной единице.

Для трёх голов ниже используем фиксированные наклоны $(1/16,1/256,1/4)$ — правило ALiBi для числа голов, не являющегося степенью двойки. Мы не подбираем их по результатам этого опыта.

<a id="section-7"></a>

## 7. T5: обучаем позиционную поправку


### Поправку можно выучить

Перенесём относительную позиционную поправку T5 в голову Годоя, сохранив остальную модель. Это не воспроизведение полной архитектуры T5. Позиционный вклад — элемент обучаемой таблицы для каждой головы. Сначала знаковое смещение превращается в номер интервала, затем соответствующая поправка прибавляется к оценке:
$$s_{ij}=q_i^\top k_j/\sqrt{d_k}+b_{h,\operatorname{bucket}(i-j)}.$$
Близкие расстояния различаются точно, дальние объединяются в логарифмически расширяющиеся интервалы. В энкодере различаются оба направления; в декодере будущие ключи закрывает отдельная маска. `relative_bias` начинается с нулей, но получает градиенты вместе с остальными параметрами.

In [ ]:
def relative_bucket(delta, bidirectional=True, num_buckets=32, max_distance=128):
    # delta = i-j; отрицательное значение означает ключ справа от запроса.
    buckets = num_buckets // 2 if bidirectional else num_buckets
    sign = (delta < 0).long() * buckets if bidirectional else torch.zeros_like(delta)
    distance = delta.abs() if bidirectional else delta.clamp_min(0)
    exact = buckets // 2
    large = exact + (torch.log(distance.clamp_min(exact).float()/exact)
                     / math.log(max_distance/exact) * (buckets-exact)).long()
    return sign + torch.where(distance < exact, distance, large.clamp_max(buckets-1))

class AttentionT5(Attention):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.bidirectional = True
        self.symmetric = False
        self.relative_bias = nn.Parameter(torch.zeros(32))

    def score_function(self, query):
        scores = super().score_function(query)
        i = torch.arange(query.size(1), device=query.device)[:, None]
        j = torch.arange(self.proj_keys.size(1), device=query.device)[None, :]
        delta = (i-j).abs() if self.symmetric else i-j
        index = relative_bucket(delta, bidirectional=self.bidirectional)
        return scores + self.relative_bias[index]

In [ ]:
distances = torch.arange(-160, 161)
fig, ax = plt.subplots(figsize=(10, 4))
ax.step(distances.numpy(), relative_bucket(distances).numpy(), where="mid")
ax.set(xlabel="Смещение i−j", ylabel="Номер интервала", title="T5: отдельные индексы для двух направлений")
fig.tight_layout()
plt.show()

У нас всего две позиции в энкодере и две в декодере. Поэтому в обучении встретятся только смещения $-1,0,1$; большинство элементов таблицы вообще не будет использовано. Рисунок объясняет правило для больших расстояний, но обучение на квадрате не проверяет поведение этих неиспользованных интервалов.

<a id="section-8"></a>

## 8. Обучение на одних и тех же квадратах


### Одинаковая сборка моделей

Сохраним обёртки `EncoderPe` и `DecoderPe` из книги. Для Sinusoidal они прибавляют PE; для остальных вариантов это тождественное преобразование. Варианты RoPE, ALiBi и T5 меняют класс головы внутри слоя самовнимания. Число голов, размерности, полносвязные слои и перекрёстное внимание одинаковы.

In [ ]:
class EncoderPe(nn.Module):
    def __init__(self, n_heads, d_model, ff_units, n_features=None, max_len=100, attention_cls=Attention, use_pe=True):
        super().__init__()
        pe_dim = d_model if n_features is None else n_features
        self.pe = PositionalEncoding(max_len, pe_dim) if use_pe else nn.Identity()
        self.layer = EncoderSelfAttn(n_heads, d_model, ff_units, n_features, attention_cls=attention_cls)

    def forward(self, query, mask=None):
        query_pe = self.pe(query)
        out = self.layer(query_pe, mask)
        return out

class DecoderPe(nn.Module):
    def __init__(self, n_heads, d_model, ff_units, n_features=None, max_len=100, attention_cls=Attention, use_pe=True):
        super().__init__()
        pe_dim = d_model if n_features is None else n_features
        self.pe = PositionalEncoding(max_len, pe_dim) if use_pe else nn.Identity()
        self.layer = DecoderSelfAttn(n_heads, d_model, ff_units, n_features, attention_cls=attention_cls)

    def init_keys(self, states):
        self.layer.init_keys(states)

    def forward(self, query, source_mask=None, target_mask=None):
        query_pe = self.pe(query)
        out = self.layer(query_pe, source_mask, target_mask)
        return out

In [ ]:
attention_classes = {"NoPE": Attention, "Sinusoidal": Attention,
                     "RoPE": AttentionRoPE, "ALiBi": AttentionALiBi, "T5": AttentionT5,
                     "T5-symmetric": AttentionT5}

def make_model(name, init_seed=43):
    torch.manual_seed(init_seed)
    cls = attention_classes[name]
    args = dict(n_heads=3, d_model=2, ff_units=10, n_features=2,
                attention_cls=cls, use_pe=(name == "Sinusoidal"))
    model = EncoderDecoderSelfAttn(EncoderPe(**args), DecoderPe(**args), input_len=2, target_len=2)
    if name == "ALiBi":
        for module in [model.encoder.layer, model.decoder.layer]:
            for head, slope in zip(module.self_attn_heads.attn_heads, [1/16, 1/256, 1/4]):
                head.slope.fill_(slope)
    if name in ("T5", "T5-symmetric"):
        for head in model.decoder.layer.self_attn_heads.attn_heads:
            head.bidirectional = False
    if name == "T5-symmetric":
        for head in model.encoder.layer.self_attn_heads.attn_heads:
            head.symmetric = True
    return model

base_state = copy.deepcopy(make_model("NoPE").state_dict())

def make_experiment(name, init_seed=43):
    model = make_model(name, init_seed)
    shared_state = make_model("NoPE", init_seed).state_dict()
    # Общие обучаемые матрицы стартуют с одинаковых значений.
    state = model.state_dict()
    state.update({key: value.clone() for key, value in shared_state.items() if key in state})
    model.load_state_dict(state)
    loss = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    sbs = StepByStep(model, loss, optimizer)
    sbs.to(device)
    sbs.set_loaders(*make_loaders())
    return sbs

n_epochs = 100
runs = {}

### Настройка и обучение

Каждый вариант получает одинаковые исходные матрицы и одинаковый порядок мини-пакетов (seed=42), 100 эпох, Adam с шагом 0.01 и MSE. У T5 дополнительно обучаются таблицы поправок, поэтому число параметров немного больше. Кривые обучения используют известную C, кривые проверки — предсказанную: разрыв между ними отражает в том числе разницу режимов.

Ниже повторяется знакомая тройка действий из книги: `StepByStep`, `train`, `plot_losses`. После кривых сразу показываем предсказанные C и D. Как у Годоя, сплошные чёрные отрезки соединяют истинные вершины, пунктирные — предсказанные; белые номера внутри чёрных кружков показывают порядок обхода. Вершины 1 и 2 (A и B) известны, вершины 3 и 4 (C и D) предсказывает модель. Никакой вариант не переобучается специально до «красивого» результата.

### NoPE

Начнём с контроля без явных позиций. Маска декодера остаётся на месте.

In [ ]:
sbs_nope = make_experiment("NoPE")
sbs_nope.train(n_epochs, seed=42)
runs["NoPE"] = sbs_nope
fig = sbs_nope.plot_losses()

In [ ]:
fig = sequence_pred(sbs_nope, full_test, test_directions)

### Sinusoidal

Теперь координаты получают добавку позиции, как в PositionalEncoding книги.

In [ ]:
sbs_sinusoidal = make_experiment("Sinusoidal")
sbs_sinusoidal.train(n_epochs, seed=42)
runs["Sinusoidal"] = sbs_sinusoidal
fig = sbs_sinusoidal.plot_losses()

In [ ]:
fig = sequence_pred(sbs_sinusoidal, full_test, test_directions)

### RoPE

В этой модели преобразованные запросы и ключи поворачиваются перед вычислением оценок.

In [ ]:
sbs_rope = make_experiment("RoPE")
sbs_rope.train(n_epochs, seed=42)
runs["RoPE"] = sbs_rope
fig = sbs_rope.plot_losses()

In [ ]:
fig = sequence_pred(sbs_rope, full_test, test_directions)

### ALiBi

Векторы не получают позиционной добавки. Меняются оценки совместимости.

In [ ]:
sbs_alibi = make_experiment("ALiBi")
sbs_alibi.train(n_epochs, seed=42)
runs["ALiBi"] = sbs_alibi
fig = sbs_alibi.plot_losses()

In [ ]:
fig = sequence_pred(sbs_alibi, full_test, test_directions)

### T5

В этой модели величины позиционных поправок определяются обучением.

In [ ]:
sbs_t5 = make_experiment("T5")
sbs_t5.train(n_epochs, seed=42)
runs["T5"] = sbs_t5
fig = sbs_t5.plot_losses()

In [ ]:
fig = sequence_pred(sbs_t5, full_test, test_directions)

<a id="section-9"></a>

## 9. Что показывает сравнение


### Те же примеры рядом

В предыдущих рисунках всегда использовались первые четыре объекта проверочной выборки. Теперь поставим первые два рядом для всех методов. Это позволяет увидеть, смещается ли одна вершина, сжимаются ли обе, либо ошибка первого шага искажает второй. Шум в C и D независим от входа, поэтому даже хороший прогноз не обязан совпасть с каждой шумной точкой.

In [ ]:
fig = plot_comparison(runs, full_test)

In [ ]:
validation_mse = {}
for name, sbs in runs.items():
    pred = torch.from_numpy(sbs.predict(source_test))
    validation_mse[name] = F.mse_loss(pred, target_test).item()
    parameters = sum(p.numel() for p in sbs.model.parameters())
    print(f"{name:12s} | параметров: {parameters:4d} | MSE C,D: {validation_mse[name]:.5f}")

### Сравнение десяти начальных состояний

Предыдущая таблица описывает один запуск с `init_seed=43`. Чтобы понять, насколько результат устойчив, повторим каждый метод с **десятью инициализациями 43–52**. Внутри каждой пары «seed — метод» общие матрицы одинаковы. Данные обучения неизменны: 256 последовательностей, seed=13; порядок мини-пакетов задаётся seed=42. Adam, шаг 0.01, 100 эпох и финальная эпоха одинаковы; лучшую эпоху по проверке не выбираем.

Для этой дополнительной оценки сгенерированы **4096 новых последовательностей**, seed=20261123. Они не участвуют в обучении; все методы оцениваются на одних примерах. MSE усредняется по двум координатам и двум предсказанным вершинам при свободной генерации. Выборка из 128 примеров выше остаётся отдельной. Здесь меняется только инициализация, поэтому разброс ниже не оценивает влияние выбора обучающей выборки.

Числа ниже — результаты выполненного сравнения на CPU (Python 3.13.13, PyTorch 2.14.0), а не демонстрационные значения. Сохраняем их явно, чтобы чтение главы не требовало заново обучать 50 моделей. Код повторения приведён ниже.

In [ ]:
# Строки: init_seed=43,…,52; столбцы: пять методов в указанном порядке.
method_names = ['NoPE', 'Sinusoidal', 'RoPE', 'ALiBi', 'T5']
seed_mse = np.array([[0.14183327555656433, 0.019070379436016083, 0.030042804777622223, 0.024099476635456085, 0.05386807769536972], [0.015698079019784927, 0.013816521503031254, 0.14895588159561157, 0.020567912608385086, 0.021872587502002716], [0.1204117089509964, 0.09335225820541382, 0.017247667536139488, 0.016943369060754776, 0.012831456027925014], [0.012313561514019966, 0.033585499972105026, 0.011957120150327682, 0.023676147684454918, 0.015272058546543121], [0.18789391219615936, 0.01929183304309845, 0.13650336861610413, 0.011965526267886162, 0.011838441714644432], [0.03418746590614319, 0.21119871735572815, 0.10372576117515564, 0.40229761600494385, 0.09941284358501434], [0.3159206211566925, 0.014621289446949959, 0.01888381317257881, 0.016488835215568542, 0.019934121519327164], [0.012697869911789894, 0.013498188927769661, 0.13302776217460632, 0.011743543669581413, 0.013120997697114944], [0.017081493511795998, 0.017206132411956787, 0.037446510046720505, 0.22911517322063446, 0.018629565834999084], [0.2522812783718109, 0.13759386539459229, 0.09064324200153351, 0.2758488655090332, 0.03588346764445305]])
for name, values in zip(method_names, seed_mse.T):
    print(f"{name:12s} | среднее {values.mean():.5f} ± {values.std(ddof=1):.5f} | медиана {np.median(values):.5f} | min–max {values.min():.5f}–{values.max():.5f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for i, values in enumerate(seed_mse.T):
    ax.scatter(i + np.linspace(-.12, .12, len(values)), values, c='k', s=35)
    ax.plot([i-.2, i+.2], [np.median(values)]*2, c='r', lw=2)
ax.axhline(.01, c='k', ls='--', lw=2, label=r"Шум целевых координат: $\sigma^2=0.01$")
ax.plot([], [], c='r', lw=2, label="Медиана десяти запусков")
ax.set(xticks=range(len(method_names)), xticklabels=method_names,
       ylabel="MSE свободной генерации", yscale="log", title="Квадрат: разброс по начальным весам")
ax.legend(loc="upper center", bbox_to_anchor=(.5, -.16), fontsize=10)
fig.tight_layout()
plt.show()

### Сначала — геометрический ориентир

Для базового квадрата с центром в начале координат противоположные вершины удовлетворяют

$$C_0=-A_0,\qquad D_0=-B_0.$$

Генератор независимо добавляет к каждой координате шум с $\sigma=0.1$. Если просто предсказывать $\hat C=-A$, $\hat D=-B$ по шумным входам, ошибка содержит два независимых шума и её ожидаемая MSE равна $2\sigma^2=0.02$. Если восстановить исходные углы округлением каждой координаты к $-1$ или $+1$, останется практически только непредсказуемый шум целевых вершин: MSE около $\sigma^2=0.01$. Это ориентир с известной геометрией генератора, а не ещё одна обученная сеть.

Следовательно, модель может решить задачу, восстановив углы и связь входа с выходом. Дальние зависимости и экстраполяция длины здесь не проверяются.

In [ ]:
evaluation_points, _ = generate_sequences(n=4096, seed=20261123)
evaluation_full = torch.tensor(np.asarray(evaluation_points)).float()
for label, pred in [("Отрицание шумного входа", -evaluation_full[:, :2]),
                    ("Восстановление углов", -evaluation_full[:, :2].sign())]:
    print(label, F.mse_loss(pred, evaluation_full[:, 2:]).item())

### Выводы по каждому способу

На десяти начальных состояниях **T5 даёт наименьшую среднюю ошибку и наименьшую максимальную ошибку среди пяти вариантов**. У Sinusoidal и T5 практически одинаковая медиана. Поэтому точнее говорить, что в этой серии T5 реже даёт большую ошибку, а не что он выигрывает в каждом запуске.

| Метод | Средняя MSE | Стандартное отклонение | Медиана | Диапазон |
|---|---:|---:|---:|---:|
| NoPE | 0.11103 | 0.11161 | 0.07730 | 0.01231–0.31592 |
| Sinusoidal | 0.05732 | 0.06843 | 0.01918 | 0.01350–0.21120 |
| RoPE | 0.07284 | 0.05531 | 0.06404 | 0.01196–0.14896 |
| ALiBi | 0.10327 | 0.14383 | 0.02212 | 0.01174–0.40230 |
| T5 | 0.03027 | 0.02758 | 0.01928 | 0.01184–0.09941 |

Стандартное отклонение описывает разброс запусков, а не погрешность среднего. Десять инициализаций и одна обучающая выборка дают локальный результат для заданного бюджета обучения. Особенно при редких больших ошибках порядок средних чувствителен к составу серии.

**Без явной позиции, NoPE.** Наибольшая средняя ошибка и большой разброс. Однако лучшие запуски близки к уровню шума. Следовательно, проблема не в принципиальной невозможности восстановить порядок: декодер отдельно получает последнюю входную вершину B и может отличить её от другой вершины в выходах энкодера. Неудачные запуски указывают на трудность обучения данной маленькой сети. Это наблюдение о сходимости, а не доказательство конкретного вида локального минимума.

**Синусоидальное кодирование Годоя.** Почти такая же медиана, как у T5, но несколько крупных ошибок повышают среднее. Позиция добавляется к входному вектору до вычисления Q, K и V, поэтому информация о номере шага проходит и в передаваемые значения. У этой сети всего две координаты представления и нет остаточного пути: такая прямая метка шага может облегчать обучение. Это объяснение устройства; её вклад отдельно проверяем ниже.

**RoPE.** Средняя ошибка выше, чем у Sinusoidal и T5, медиана заметно выше. При удачном старте метод также достигает низкой ошибки. Здесь каждая голова двумерна: имеется всего одна частота, а расстояния равны 0 или 1. Метод меняет оценки Q–K, но не добавляет позицию в V. Его преимущества на длинных последовательностях в этом опыте проявиться не могут; по результатам нельзя заключить, что RoPE вообще хуже синусоидального кодирования. В части запусков ошибка первой предсказанной вершины усиливается на следующем шаге.

**ALiBi.** Низкая медиана соседствует с самым большим разбросом и худшим отдельным запуском. На двух позициях линейная поправка задаёт только предпочтение собственной позиции перед соседней. Даже у головы с наибольшим наклоном множитель отношения весов равен $\exp(-1/4)\approx0.779$; у остальных он ещё ближе к единице. Более того, симметричная матрица расстояний энкодера из двух позиций не меняется при перестановке A и B: эта адаптация не задаёт различие «раньше/позже» в энкодере. Это ограничение сигнала, а не ошибка реализации. Данные не дают основания считать ALiBi устойчивым улучшением NoPE для этой постановки.

**Обучаемая поправка T5.** Лучшая средняя ошибка в этой серии; типичный результат сопоставим с синусоидами. В отличие от ALiBi, знаковая таблица энкодера различает смещения −1 и +1, а величина поправки подбирается по задаче. На короткой последовательности достаточно нескольких используемых элементов таблицы. Отдельное сравнение ниже не подтверждает устойчивого преимущества знака: симметричная обучаемая таблица достигает близкого среднего и выигрывает на шести из десяти начальных состояний. Дополнительные параметры и нулевая инициализация таблицы также являются частью условий этого сравнения.

### Где возникает ошибка: первая или вторая вершина

| Метод | MSE C | MSE D при генерации | MSE D с истинной C |
|---|---:|---:|---:|
| NoPE | 0.10836 | 0.11370 | 0.06613 |
| Sinusoidal | 0.05514 | 0.05950 | 0.05499 |
| RoPE | 0.06523 | 0.08046 | 0.06901 |
| ALiBi | 0.10263 | 0.10391 | 0.07232 |
| T5 | 0.03121 | 0.02932 | 0.02621 |

Последний столбец — диагностический расчёт с истинной C, он не участвует в рейтинге свободной генерации. Разность двух столбцов D показывает влияние замены истинной C предсказанной; она не обязана быть положительной для каждой отдельной модели. Ошибки уже велики и в C: объяснить все неудачи одним накоплением ошибки декодера нельзя.

### Контроль масштаба и места добавления синусоид

Сравним первые пять инициализаций 43–47. В вариантах «только энкодер» и «только декодер» множитель $\sqrt{2}$ сохранён в обеих частях, но позиционный вектор добавляется только в указанной части. Данные, оптимизатор и бюджет прежние.

| Вариант | Средняя MSE | Стандартное отклонение | Медиана | Диапазон |
|---|---:|---:|---:|---:|
| NoPE | 0.09563 | 0.07841 | 0.12041 | 0.01231–0.18789 |
| Только масштаб $\sqrt{2}$, без PE | 0.10832 | 0.13411 | 0.01937 | 0.01172–0.30305 |
| Синусоиды как у Годоя | 0.03582 | 0.03299 | 0.01929 | 0.01382–0.09335 |
| Синусоиды без масштабирования | 0.17489 | 0.12295 | 0.16501 | 0.02423–0.32387 |
| Синусоиды только в энкодере | 0.03427 | 0.02758 | 0.02033 | 0.01716–0.08262 |
| Синусоиды только в декодере | 0.11872 | 0.11208 | 0.10203 | 0.01326–0.26645 |

**Что следует из контролей.** Одно масштабирование без позиции не улучшило среднюю ошибку NoPE. Но удаление множителя $\sqrt{2}$ из синусоидального варианта сильно ухудшило результат в этой серии. Значит, нельзя приписывать всё отличие только наличию позиции или только масштабу: здесь существенна их комбинация и её взаимодействие с оптимизацией.

Добавление синусоид только в энкодер сохранило близкую среднюю ошибку полного варианта; добавление только в декодер оказалось значительно менее устойчивым. Это поддерживает объяснение через различение двух входных вершин в энкодере. В этой серии добавление позиции ещё и в декодер не дало устойчивого преимущества. Пять seed — небольшой контрольный опыт; это не доказательство бесполезности позиции в декодере для других задач.


### Достаточно ли просто обучать NoPE дольше

Для тех же первых пяти seed продолжили обучение NoPE ещё 200 эпох с тем же шагом Adam. Порядок мини-пакетов при продолжении заново начинается с seed=42; это явно отдельный контроль бюджета, а не часть рейтинга 100 эпох.

| Начальное состояние | MSE после 100 эпох | MSE после 300 эпох |
|---|---:|---:|
| 43 | 0.14183 | 0.14237 |
| 44 | 0.01570 | 0.01797 |
| 45 | 0.12041 | 0.10294 |
| 46 | 0.01231 | 0.01082 |
| 47 | 0.18789 | 0.01065 |

Дополнительные эпохи улучшают часть запусков, но не исправляют все неудачные инициализации. Гипотеза «NoPE всегда проигрывает лишь потому, что ему нужно немного больше эпох» этой проверкой не подтверждается. Отдельная настройка шага, ширины или архитектуры может изменить результат; здесь она не проводилась.

Для демонстрации на квадрате полезно оставить синусоидальный вариант книги как исходный образец и T5 как вариант с наименьшей средней ошибкой и разбросом проведённой серии. NoPE показывает, что явная позиция не обязательна для решения этой задачи; RoPE и ALiBi показывают разные способы изменить оценки совместимости, но сами по себе не гарантируют лучшую сходимость.


### Что помогает T5: направление или обучаемая поправка?

В энкодере две позиции. Знаковая таблица задаёт три поправки: для собственной позиции, ключа слева и ключа справа. Симметричная использует одну поправку для обоих соседей:

$$B_{\mathrm{signed}}=\begin{pmatrix}b_0&b_{-1}\\b_{+1}&b_0\end{pmatrix},\qquad
B_{\mathrm{sym}}=\begin{pmatrix}b_0&b_1\\b_1&b_0\end{pmatrix}.$$

Обе таблицы обучаются с нулевой инициализации. Меняется только индексация таблицы **энкодера**: `relative_bucket(i-j)` или `relative_bucket(abs(i-j))`. В причинном слое самовнимания декодера ключи справа запрещены, поэтому его таблицу оставляем прежней. Перекрёстное внимание также не меняем. Число выделенных параметров одинаково; используемых элементов таблицы энкодера на голову три и два соответственно. Это неизбежное ограничение симметрии, а не увеличение размера сети.

Повторяем 100 эпох на CPU для тех же десяти seed 43–52, 256 обучающих последовательностей (seed=13), одинаковых исходных общих весов, Adam и порядка мини-пакетов (seed=42). Проверяем на тех же 4096 новых последовательностях (seed=20261123). Знаковые результаты при повторном запуске совпали с предыдущей серией. Каждая строка таблицы ниже — **пара с одинаковой инициализацией**, а не два независимо выбранных запуска.

| Seed | Знаковая T5, MSE | Симметричная T5, MSE | Симметричная − знаковая |
|---|---:|---:|---:|
| 43 | 0.05387 | 0.14321 | +0.08935 |
| 44 | 0.02187 | 0.01425 | -0.00762 |
| 45 | 0.01283 | 0.01327 | +0.00044 |
| 46 | 0.01527 | 0.01262 | -0.00265 |
| 47 | 0.01184 | 0.01158 | -0.00026 |
| 48 | 0.09941 | 0.02720 | -0.07221 |
| 49 | 0.01993 | 0.01468 | -0.00526 |
| 50 | 0.01312 | 0.01136 | -0.00176 |
| 51 | 0.01863 | 0.01872 | +0.00009 |
| 52 | 0.03588 | 0.06280 | +0.02692 |

| Таблица | Средняя MSE | Стандартное отклонение | Медиана |
|---|---:|---:|---:|
| Знаковая | 0.03027 | 0.02758 | 0.01928 |
| Симметричная | 0.03297 | 0.04176 | 0.01446 |

Средняя парная разность равна **+0.00270**: знак положительный, когда симметричный вариант хуже. Но симметричная таблица выигрывает **6 из 10** пар; большие разнонаправленные отличия при seed=43 и seed=48 сильно влияют на среднее. Ни число побед, ни десять запусков на одной обучающей выборке не дают основания объявить один вариант универсально лучше.


In [ ]:
# Зафиксированные результаты фактического опыта на CPU, PyTorch 2.14.0.
t5_signed_mse = np.array([0.05386807769536972, 0.021872587502002716, 0.012831456027925014, 0.015272058546543121, 0.011838441714644432, 0.09941284358501434, 0.019934121519327164, 0.013120997697114944, 0.018629565834999084, 0.03588346764445305])
t5_symmetric_mse = np.array([0.14321354031562805, 0.014249276369810104, 0.013270851224660873, 0.012620354071259499, 0.011577228084206581, 0.027203798294067383, 0.01467840000987053, 0.011356255039572716, 0.01871752366423607, 0.06280077248811722])
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for index, seed in enumerate(range(43, 53)):
    axes[0].plot([0, 1], [t5_signed_mse[index], t5_symmetric_mse[index]],
                 color="black", marker="o", alpha=.5)
axes[0].set_xticks([0, 1], ["Знаковая", "Симметричная"])
axes[0].set_yscale("log")
axes[0].set_ylabel("MSE, логарифмическая шкала")
axes[0].set_title("Линия соединяет одинаковый seed")
delta = t5_symmetric_mse - t5_signed_mse
axes[1].bar(np.arange(43, 53), delta, color="black", alpha=.7)
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_xticks(np.arange(43, 53))
axes[1].set_xlabel("Seed начальных весов")
axes[1].set_ylabel("MSE симметричной − MSE знаковой")
axes[1].set_title("Ниже нуля: симметричная лучше")
fig.tight_layout()
plt.show()


**Вывод.** В этой задаче устойчивое преимущество T5 не объясняется необходимостью различать направление смещения. Симметричная обучаемая поправка достаточна для результатов того же порядка. По сравнению с NoPE её средняя MSE также существенно ниже (0.03297 против 0.11103), однако в обоих вариантах T5 обучаются и таблицы декодера. Поэтому опыт поддерживает объяснение через обучаемую поправку в данной архитектуре, но не выделяет отдельно причинный вклад таблицы энкодера и таблицы декодера. Для такого вывода нужен ещё контроль с выключенной таблицей в каждой из частей.

Симметричный энкодер из двух позиций сохраняет перестановочную эквивариантность: перестановка A и B переставляет его выходы. Это проверено после обучения на всех десяти seed. Но полная сеть получает B отдельно на вход декодера, поэтому симметрия энкодера не делает задачу неразрешимой. Именно здесь важно отличать свойство одного слоя от свойства всей модели.

Проверки реализации: исходные веса и предсказания двух вариантов совпадают до обучения; в симметричной таблице энкодера градиенты появляются только в корзинах 0 и 1; знаковая использует 0, 1 и 17. Декодер и причинная маска не изменены. Воспроизведение полного парного опыта занимает 20 обучений; следующая ячейка только определяет функцию. Для запуска вызовите `repeat_t5_pair()`.


In [ ]:
def repeat_t5_pair(seeds=range(43, 53), epochs=100):
    points, _ = generate_sequences(n=4096, seed=20261123)
    data = torch.tensor(np.asarray(points)).float()
    rows = []
    for seed in seeds:
        pair = []
        for name in ("T5", "T5-symmetric"):
            experiment = make_experiment(name, init_seed=seed)
            experiment.train(epochs, seed=42)
            pred = torch.from_numpy(experiment.predict(data[:, :2]))
            pair.append(F.mse_loss(pred, data[:, 2:]).item())
        rows.append(pair)
    return np.asarray(rows)


### Как повторить сравнение

Функция ниже воспроизводит основную серию из 50 обучений. При обычном выполнении главы она только определяется. Для отдельного повторения вызовите `repeat_comparison()`; это дополнительный вычислительный опыт. Он использует выбранное в начале устройство, поэтому числа на MPS могут отличаться от сохранённого CPU-запуска. На этой машине серия CPU заняла около четырёх минут вместе с дополнительными контролями.

In [ ]:
def repeat_comparison(seeds=range(43, 53), epochs=100):
    points, _ = generate_sequences(n=4096, seed=20261123)
    data = torch.tensor(np.asarray(points)).float()
    result = []
    for init_seed in seeds:
        row = []
        for name in method_names:
            experiment = make_experiment(name, init_seed=init_seed)
            experiment.train(epochs, seed=42)
            pred = torch.from_numpy(experiment.predict(data[:, :2]))
            row.append(F.mse_loss(pred, data[:, 2:]).item())
        result.append(row)
    return np.asarray(result)

### Что проверено в реализации

- **Исходная модель.** NoPE и Sinusoidal сверены с классами `Chapter09.ipynb` Годоя: при одинаковой инициализации выходы обучения и генерации совпадают точно. Сохранены три головы по две координаты, слой ReLU, отсутствие residual/LayerNorm и исходный шаг Adam.
- **Позиции при генерации.** На каждом шаге повторно обрабатывается весь префикс декодера: сначала B, затем B и предсказанная C. Номера позиций — 0, затем 0 и 1. Здесь нет KV-cache и нет ошибки сброса позиции нового токена к нулю.
- **RoPE.** Вращаются аффинно преобразованные Q и K, V не вращается. Оценки совпали с независимым расчётом через комплексные числа на CPU и MPS; сохранение нормы и инвариантность к общему сдвигу проверены отдельно.
- **ALiBi.** Штраф добавляется к уже масштабированной оценке. Наклоны для трёх голов — $1/16,1/256,1/4$, по правилу авторской реализации. В причинном декодере $-|i-j|=-(i-j)$ для разрешённых пар. Симметричный штраф в двунаправленном энкодере — наша явно указанная адаптация.
- **T5.** Знаковые интервалы сверены с официальной функцией; в нашем коде `delta=i-j`, в исходнике аргумент равен `j-i`. Проверены градиенты таблиц и направления смещений. При нулевой таблице модель совпадает с NoPE; то же верно для ALiBi с нулевыми наклонами. Используется позиционная поправка T5 внутри головы Годоя: масштаб $1/\sqrt{d_k}$ и остальные слои остаются от учебной модели; это не полная архитектура T5.
- **Отсутствие утечки.** Изменение будущих C/D не меняет первую предсказанную вершину; D не попадает во вход декодера при обучении. Маски дают нулевые веса запрещённым ключам. В оценке обе вершины генерируются без истинной C.
- **Повторяемость.** `init_seed` теперь передаётся явно в `make_model` и `make_experiment`. Ранее внутри фабрики стояла константа 43: внешнего вызова `torch.manual_seed` было бы недостаточно для сравнения инициализаций. Проверено, что разные seed дают разные веса, а у пяти методов внутри одного seed общие веса совпадают.

Ошибок формул, масок и сдвига последовательности в проверенных режимах не обнаружено. Различия качества нельзя объяснять одинаковым числом параметров: T5 содержит 192 дополнительных параметра. На этой длине используются лишь 15 элементов таблиц — три смещения в каждой из трёх голов энкодера и два в каждой голове декодера.

Первоисточники проверки: [RoFormer, §3.2](https://arxiv.org/abs/2104.09864), [авторский код ALiBi](https://github.com/ofirpress/attention_with_linear_biases/blob/master/fairseq/models/transformer.py), [официальная функция интервалов T5](https://github.com/tensorflow/mesh/blob/master/mesh_tensorflow/transformer/transformer_layers.py).

### Заглянем в головы

Для рисунка весов используем один и тот же точный квадрат и вход декодера $(B,C)$ с известной C. Это режим teacher forcing для анализа, не график свободной генерации. Верхний правый элемент должен быть нулём у любой модели из-за маски. Меняться могут веса второй строки и распределения в энкодере.

Веса внимания сами по себе не являются оценкой качества прогноза: после них ещё работают объединение голов, перекрёстное внимание и полносвязная сеть.

In [ ]:
for name, sbs in runs.items():
    fig = plot_attention(sbs.model, full_seq, kind="decoder")
    fig.suptitle(name + ": самовнимание декодера при известной C", fontsize=14)
    fig.tight_layout(rect=(0, 0, 1, .85))
    plt.show()

### Какая поправка действительно выучилась

У T5 посмотрим только на используемые смещения. В энкодере есть левый и правый сосед; в декодере — текущая и предыдущая позиции. Полезны именно **разности** поправок: одинаковая константа во всей строке исчезнет после softmax.

In [ ]:
for location, module, delta in [
    ("Энкодер", runs["T5"].model.encoder.layer, [-1, 0, 1]),
    ("Декодер", runs["T5"].model.decoder.layer, [0, 1])]:
    print(location, "смещения:", delta)
    for h, head in enumerate(module.self_attn_heads.attn_heads):
        index = relative_bucket(torch.tensor(delta, device=device), head.bidirectional)
        print("Голова", h+1, head.relative_bias[index].detach().cpu().numpy().round(4))

Здесь сравниваются пять способов включить позицию в одну маленькую учебную сеть. Основные иллюстрации показывают один seed; дополнительная серия сравнивает десять начальных состояний на одной фиксированной обучающей выборке. Кроме того, последовательности длины два не проверяют дальние зависимости, частоты RoPE на больших расстояниях или экстраполяцию длины. Синусоидальный вариант сохраняет масштабирование входов книги; T5 добавляет параметры. Поэтому опыт объясняет устройство моделей и их обучение, но не выбирает лучший позиционный механизм для LLM.

<a id="section-10"></a>

## 10. NoPE и информация в причинной маске


### Маска тоже меняет вычисление

До этого NoPE означало отсутствие явных позиционных преобразований в нашем encoder–decoder. В исследованиях NoPE часто рассматривают причинный decoder-only Transformer. Причинная маска выделяет разные префиксы: позиции доступны один, два, три и далее ключей.

Рассмотрим отдельную конструкцию из обсуждения: последовательность $(\mathrm{BOS},A,B,C)$. BOS — выделенный маркер начала, **не вершина квадрата**. Пусть все разрешённые оценки равны нулю, а одна координата значений равна 1 у BOS и 0 у остальных элементов. Тогда веса равномерны внутри доступного префикса и выход этой координаты равен $1/(i+1)$.

In [ ]:
scores = torch.zeros(1, 4, 4)
mask = torch.ones(1, 4, 4, dtype=torch.bool).tril()
marker_values = torch.tensor([1., 0., 0., 0.]).view(1, 4, 1)
causal_alphas = scores.masked_fill(~mask, -torch.inf).softmax(-1)
position_signal = torch.bmm(causal_alphas, marker_values)
position_signal.squeeze(), (1 / position_signal - 1).squeeze()

In [ ]:
fig = plot_matrix(causal_alphas[0], "NoPE: равные оценки и причинная маска", ["BOS", "A", "B", "C"])

Получился позиционный сигнал без PE, но в конструкции участвуют и маска, и различимый BOS. Если все значения одинаковы, усреднение одинаковых чисел снова даёт одно и то же число. А без причинной маски каждая строка увидит один и тот же набор из четырёх значений.

In [ ]:
identical_values = torch.ones_like(marker_values)
without_marker = torch.bmm(causal_alphas, identical_values)
without_mask = torch.bmm(scores.softmax(-1), marker_values)
without_marker.squeeze(), without_mask.squeeze()

Это демонстрация возможного вычисления, а не наблюдение того, что наши обученные модели самостоятельно построили счётчик. BOS не добавлялся в обучающую выборку квадратов. Результаты NoPE из [Kazemnejad et al., NeurIPS 2023](https://arxiv.org/abs/2305.19466) относятся к другой архитектуре и набору задач; переносить их как гарантию на этот опыт или на произвольный длинный контекст нельзя.

<a id="section-11"></a>

## 11. Другие относительные представления


### Где находятся Shaw, Transformer-XL и DeBERTa

За рамками проведённого сравнения остаются другие способы изменить оценку совместимости. Но для обучения выше выбраны пять сопоставимых вариантов, соответствующих задаче этой главы. Остальные формулы полезно читать как продолжение, не как ещё три полностью реализованные архитектуры.

У Shaw к ключу добавляется обучаемое представление относительного смещения: $q_i^\top(k_j+r^K_{i-j})$. В исходной статье есть и относительная добавка к значениям. У Transformer-XL оценка раскладывается на четыре содержательных и позиционных вклада; вместе с ней используется память предыдущих сегментов. У DeBERTa два перекрёстных вклада разделены:
$$q_i^\top k_j+q_i^\top r^K_{i-j}+k_j^\top r^Q_{j-i}.$$
В последнем слагаемом направление смещения обратное. Масштабирование и преобразования относительных векторов нужно восстановить по статье. Подстановка этих выражений в `Attention.score_function` — ещё не воспроизведение целой модели.

<a id="section-12"></a>

## 12. Источники и устройство файлов


### Что сохранено от книги

Генератор `generate_sequences`, данные и seed, разбиение 2→2, имена `full_seq`, `source_seq`, `target_seq`, `shifted_seq`, интерфейс голов, структура энкодера/декодера, `StepByStep`, MSE, Adam и 100 эпох. Добавлены выбор типа головы, одинаковые исходные общие веса, MPS, русские подписи, SVG/Retina и новые позиционные механизмы. Код Годоя используется по [лицензии MIT](position_square/LICENSE-Godoy.txt); в исходных классах сохранены комментарии автора.

### Первоисточники

- [Godoy. Chapter09.ipynb: Sequence-to-Sequence и Positional Encoding](https://github.com/dvgodoy/PyTorchStepByStep/blob/master/Chapter09.ipynb).
- [Su et al. RoFormer: Enhanced Transformer with Rotary Position Embedding](https://arxiv.org/abs/2104.09864), §3.2.
- [Press et al. Train Short, Test Long: Attention with Linear Biases Enables Input Length Extrapolation](https://arxiv.org/abs/2108.12409), §3.
- [Raffel et al. Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer](https://www.jmlr.org/papers/v21/20-074.html), §2.1; [Mesh TensorFlow: relative position buckets](https://github.com/tensorflow/mesh/blob/master/mesh_tensorflow/transformer/transformer_layers.py).
- [Kazemnejad et al. The Impact of Positional Encoding on Length Generalization in Transformers](https://arxiv.org/abs/2305.19466), раздел о NoPE и приложение C.
- [Shaw et al. Self-Attention with Relative Position Representations](https://aclanthology.org/N18-2074/).
- [Dai et al. Transformer-XL](https://aclanthology.org/P19-1285/), §3.3.
- [He et al. DeBERTa](https://arxiv.org/abs/2006.03654), §3.1.

Для запуска в другом месте перенесите ноутбук вместе с папкой `position_square`. Эта версия не требует доступа к локальной копии книги, Google Drive или загрузки весов модели.